Python 3.11.9

In [ ]:
#!pip install pandas numpy matplotlib scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

**Cargo el fichero en formato excel:**

In [ ]:
#!pip install openpyxl

In [ ]:
#tengo el fichero en la misma carpeta
original_data = pd.read_excel('_RIRS_litotricia_ID_estudio_original_6MAYO.xlsx')
patient_data = original_data.copy()
patient_data.head()

In [ ]:
N_pat, N_columns = patient_data.shape
print(f"Number of patients: {N_pat}")
print(f"Number of columns: {N_columns}")
patient_data.dtypes

# Primeros ajustes de formato

Cambio de nombres de columnas con error en codificacion:

In [ ]:
patient_data.columns.values.tolist()

In [ ]:
patient_data.rename(columns={"radiografÃƒÂ\xada": "radiografia"}, inplace=True)

In [ ]:
patient_data.columns.values.tolist()[30:]

Quito decimal sobrante de variables categoricas:

In [ ]:
# quitar decimal sobrante de variables categoricas
categorias = ['Genero','Obesidad', 'Diabetes', 'HTA','Dislipemia', 'Fumador_tabaco', 'ASA',
              'ANOMALIAANATÓMICA','TIPOANOMALIA','LOCAL_COD','LOCALIZACIÓN','GRUPO',
              'cultivo','Cultresul','Microorganismo', 'GUY_SCORE',
              'CATETERJJPREVIO', 'Dilatacion_ureteral', 'Vaina_acceso', 'Calibre_vaina',
              'DERIVACIONPOSTERIOR', 'PULS',
              'MOTIVO', 'tipotto', 'lado','Complicaciones','tipocomplicacion','complicaciones_intra',
              'complicaciones_post','trasfusion','reintervcomplic','embolizacion','SANGRADO','PERFORACION',
              'INFECCION','SEPSIS','HEMATURIA','HEMATOMA', 'IngresoSiNo',
              'urgencias','REingreso','RESUELTOEN6m']	#'Compli_Post_Recod','Complic_Intra_Recodif'
patient_data[categorias] = patient_data[categorias].astype('Int64')
patient_data.head()

Redondear el procentaje de neutrofilos:

In [ ]:
patient_data['ProcentajeNeutrofilos'] = (patient_data['NeutrófilosTotales'] / patient_data['LeucocitosTotales']) * 100

In [ ]:
patient_data['ProcentajeNeutrofilos'] = patient_data['ProcentajeNeutrofilos'].round(decimals = 2)
patient_data.head()

# Identificación de errores y su eliminación

**Elección de las variables de interes**

In [ ]:
patient_data.columns

In [ ]:
# Crear un nuevo DataFrame con columnas específicas
columnas_caract_antes = ['codigo', 'DIAGNÓSTICO', 'Genero', 'lado',
       'cultivo', 'Cultresul',
       'Microorganismo', 'altura', 'peso', 'BMI',  'Diabetes',
       'HTA', 'Dislipemia', 'Fumador_tabaco', 'ASA', 'ANOMALIAANATÓMICA',
       'TIPOANOMALIA', 'tipotto', 'NÚMEROLITIASIS', 'TAMAÑOLITIASIS',
       'VOLUMEN', 'UH', 'LOCAL_COD', 'LOCALIZACIÓN', 'GRUPO', 'GUY_SCORE','EDAD'] #'Obesidad',
columnas_quir=['CATETERJJPREVIO', 'Dilatacion_ureteral', 'Vaina_acceso',
       'Calibre_vaina', 'DERIVACIONPOSTERIOR', 'PULS']
columnas_analitica=['Procalcitonina', 'LeucocitosTotales', 'ProcentajeNeutrofilos'] #+['ProteinaCReactiva']
columnas_complicaciones = ['Complicaciones','tipocomplicacion','complicaciones_intra',
              'complicaciones_post','trasfusion','complicaclavien','reintervcomplic','embolizacion','SANGRADO','PERFORACION',
              'INFECCION','SEPSIS','HEMATURIA','HEMATOMA']+['IngresoSiNo'] #'PERDIDAACCESO',
etiqueta_a_predecir = ['RESUELTOEN6m']
## para_matriz = ['ComplInfSeps','ComplHemo']
#SELECCION DE COLUMNAS
selected_columns = columnas_caract_antes+columnas_quir+columnas_analitica+etiqueta_a_predecir+columnas_complicaciones
patient_data = patient_data[selected_columns]
patient_data.head()

### Arreglo preliminar de unas variables

**Umbralización de variables analíticas:**

In [ ]:
patient_data['Procalcitonina'] = np.where(
    patient_data['Procalcitonina'].notna(),
    np.where(patient_data['Procalcitonina'] >= 0.5, 1, 0),
    np.nan
)

patient_data['LeucocitosTotales'] = np.where(
    patient_data['LeucocitosTotales'].notna(),
    (
        (patient_data['LeucocitosTotales'] >= 15) |
        (patient_data['LeucocitosTotales'] <= 4.5)
    ).astype(int),
    np.nan
)

patient_data['ProcentajeNeutrofilos'] = np.where(
    patient_data['ProcentajeNeutrofilos'].notna(),
    (patient_data['ProcentajeNeutrofilos'] > 80).astype(int),
    np.nan
)

**calcular BMI donde es nan pero se dispone de altura y peso**

In [ ]:
import numpy as np

# guardar cuántos BMI faltaban antes
bmi_missing_before = patient_data['BMI'].isna().sum()

# condición:
# BMI vacío + altura disponible + peso disponible
mask = (
    patient_data['BMI'].isna() &
    patient_data['altura'].notna() &
    patient_data['peso'].notna()
)

# calcular BMI
patient_data.loc[mask, 'BMI'] = (
    patient_data.loc[mask, 'peso'] /
    ((patient_data.loc[mask, 'altura'] / 100) ** 2)
)

# opcional: redondear
patient_data['BMI'] = patient_data['BMI'].round(2)

# cuántos valores se recuperaron
bmi_missing_after = patient_data['BMI'].isna().sum()
recuperados = bmi_missing_before - bmi_missing_after

print(f"Valores de BMI recuperados: {recuperados}")

### Prospección de valores anómalos en variables númericas (aunque en este caso serían de interes)

In [ ]:
numericas = ['BMI','NÚMEROLITIASIS', 'TAMAÑOLITIASIS','VOLUMEN', 'UH','EDAD'] #+['ProteinaCReactiva']
# binarias
# catgeroicas nominales
# categoricas ordinales

In [ ]:
features = numericas
fig, axes = plt.subplots(nrows=len(features), ncols=3, figsize=(12, 3 * len(features)))
fig.suptitle("Distribuciones de caracteristicas numericas", fontsize=16)

plot_types = [
    ("Boxplot", lambda ax, data: ax.boxplot(data)),
    ("Histogram", lambda ax, data: ax.hist(data, bins=20, edgecolor='black')),
    ("Violin Plot", lambda ax, data: ax.violinplot(data))
]

for row_idx, feature in enumerate(features):
    column = patient_data[feature].dropna().values  # Convertimos en array 1D sin NaNs
    for col_idx, (title, plot_func) in enumerate(plot_types):
        ax = axes[row_idx, col_idx]
        plot_func(ax, column)  # Llamamos a la función de ploteo
        if col_idx == 0:  
            ax.set_ylabel(feature)  # Etiquetamos la fila con la variable
        ax.set_title(title)  # Ponemos el tipo de gráfico como título

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

Corregir BMI raro

In [ ]:
# máscara: BMI > 60 y con altura/peso disponibles
mask = (
    (patient_data['BMI'] > 60) &
    patient_data['altura'].notna() &
    patient_data['peso'].notna()
)

# cuántos valores se van a recalcular
n_recalculados = mask.sum()

# eliminar BMI previo
patient_data.loc[mask, 'BMI'] = pd.NA

# recalcular BMI
patient_data.loc[mask, 'BMI'] = (
    patient_data.loc[mask, 'peso'] /
    ((patient_data.loc[mask, 'altura'] / 100) ** 2)
)

# redondear
patient_data['BMI'] = patient_data['BMI'].round(2)

print(f"BMI recalculados: {n_recalculados}")

# segunda limpieza: cualquier BMI que siga siendo > 60 se invalida
mask_invalidos = patient_data['BMI'] > 60
n_eliminados = mask_invalidos.sum()

patient_data.loc[mask_invalidos, 'BMI'] = pd.NA

print(f"BMI eliminados por seguir >60: {n_eliminados}")

Volumen

In [ ]:
print(patient_data['VOLUMEN'].isna().sum())
patient_data = patient_data.drop(columns=['VOLUMEN'])
numericas.remove('VOLUMEN')
numericas

Acortar pacientes pedaitricos:

In [ ]:
len(patient_data)

In [ ]:
# Filtrar pacientes menores de 18
menores_18 = patient_data[patient_data["EDAD"] < 18]

# Agrupar y contar cuántas muestras hay por cada edad
desglose_edades = menores_18["EDAD"].value_counts().sort_index()

print("Desglose de muestras por edad (menores de 18):")
print(desglose_edades)

In [ ]:
patient_data[patient_data["EDAD"] < 18].sum

In [ ]:
# Eliminar pacientes menores de 18 años
patient_data = patient_data[patient_data["EDAD"] >= 18].reset_index(drop=True)
N_pat, N_columns = patient_data.shape
print(f"Number of patients: {N_pat}")

In [ ]:
features = numericas

fig, axes = plt.subplots(nrows=len(features), ncols=2, figsize=(10, 3 * len(features)))
fig.suptitle("Distribuciones de caracteristicas numericas", fontsize=16)

plot_types = [
    ("Boxplot", lambda ax, data: ax.boxplot(data)),
    ("Histogram", lambda ax, data: ax.hist(data, bins=20, edgecolor='black'))
]

for row_idx, feature in enumerate(features):
    column = patient_data[feature].dropna().values
    for col_idx, (title, plot_func) in enumerate(plot_types):
        ax = axes[row_idx, col_idx]
        plot_func(ax, column)
        if col_idx == 0:
            ax.set_ylabel(feature)
        ax.set_title(title)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
patient_data.loc[patient_data['NÚMEROLITIASIS'] == 0, 'NÚMEROLITIASIS'] = np.nan

In [ ]:
patient_data['NÚMEROLITIASIS'].unique()

In [ ]:
features = numericas  # ['BMI','NÚMEROLITIASIS', 'TAMAÑOLITIASIS','VOLUMEN', 'UH','EDAD'] 

# Diccionario de unidades
units = {
    'BMI': 'kg/m²',
    'TAMAÑOLITIASIS': 'mm',
    'NÚMEROLITIASIS': '',
    'UH': 'UH',
    'EDAD': 'años'
}

fig, axes = plt.subplots(
    nrows=len(features),
    ncols=2,
    figsize=(12, 3 * len(features))
)

fig.suptitle(
    "Distribuciones de características numéricas",
    fontsize=16
)

plot_types = [
    ("Boxplot", lambda ax, data: ax.boxplot(data)),
    ("Histogram", lambda ax, data: ax.hist(data, bins=20, edgecolor='black'))
]

for row_idx, feature in enumerate(features):

    column = patient_data[feature].dropna().values
    unit = units.get(feature, '')

    # Etiqueta completa
    label = f"{feature} ({unit})" if unit else feature

    for col_idx, (title, plot_func) in enumerate(plot_types):

        ax = axes[row_idx, col_idx]

        plot_func(ax, column)

        if col_idx == 0:
            ax.set_ylabel(label)

        ax.set_title(f"{title} - {label}")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# filtrar muestras con BMI > 50
df_bmi50 = patient_data[patient_data['BMI'] > 50]
# ver todas las columnas (o las que te interesen)
df_bmi50

In [ ]:
patient_data[patient_data['BMI'] < 15]

In [ ]:
patient_data.columns

In [ ]:
patient_data.loc[patient_data['NÚMEROLITIASIS'] > 25, ['NÚMEROLITIASIS', 'TAMAÑOLITIASIS', 'GUY_SCORE','GRUPO']]

In [ ]:
patient_data.loc[patient_data['TAMAÑOLITIASIS'] > 40, ['NÚMEROLITIASIS', 'TAMAÑOLITIASIS', 'GUY_SCORE','GRUPO']]

In [ ]:
patient_data = patient_data.drop(columns=['altura','peso'])

#### identificación de errores en variables categoricas

In [ ]:
categoricas = [col for col in patient_data.columns if col not in numericas and col not in ['codigo','RESUELTOEN6m','ComplInfSeps','ComplHemo'] and col not in columnas_complicaciones]
categoricas

In [ ]:
#mirar frecuencias de los valores
# mirar valores diferentes del numero, como el fichero originalmente es de spss

In [ ]:
import matplotlib.pyplot as plt
for col in categoricas:
    print(f"\n--- {col} ---")
    
    # frecuencias en %
    freq = patient_data[col].value_counts(dropna=False, normalize=True) * 100
    print(freq.round(2))

    # pie plot
    plt.figure()
    freq.plot.pie(autopct='%1.1f%%', startangle=90)
    plt.title(f"Distribución de {col}")
    plt.ylabel("")  # quita etiqueta del eje
    plt.show()

In [ ]:
patient_data = patient_data.drop(columns=['DIAGNÓSTICO'])
categoricas.remove('DIAGNÓSTICO')

In [ ]:
mask_invalidos = patient_data['GRUPO'] == 7
n_eliminados = mask_invalidos.sum()
patient_data.loc[mask_invalidos, 'GRUPO'] = pd.NA
print(f"GRUPO = 7: {n_eliminados}")

In [ ]:
mask_invalidos = patient_data['LOCALIZACIÓN'] > 4
n_eliminados = mask_invalidos.sum()
patient_data.loc[mask_invalidos, 'LOCALIZACIÓN'] = pd.NA
print(f"LOCALIZACIÓN == 5,6,7,8: {n_eliminados}")

In [ ]:
#!pip install seaborn

# Arreglo de <N/A>

In [ ]:
patient_data.head()

In [ ]:
patient_data.iloc[:,20:]

In [ ]:
patient_data = patient_data.replace({pd.NA: np.nan})

In [ ]:
# todo el DataFrame
patient_data = patient_data.astype('object')
patient_data = patient_data.applymap(lambda x: np.nan if pd.isna(x) else x)

In [ ]:
patient_data.head()

# EDA

In [ ]:
patient_data[numericas].describe().T

In [ ]:
import seaborn as sns

# Calcular la matriz de correlación solo con variables numéricas
correlacion = patient_data[numericas].corr()
# Visualizar
plt.figure(figsize=(12, 8))
sns.heatmap(correlacion, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación')
plt.tight_layout()
plt.show()

In [ ]:

mapeos = {
    'RESUELTOEN6m':         {0: 'No', 1: 'Sí'},
    'Genero':               {0: 'Hombre', 1: 'Mujer'},
    'Obesidad':             {0: 'No', 1: 'Sí'},
    'Diabetes':             {0: 'No', 1: 'DM insulina', 2: 'DM sin insulina'},
    'HTA':                  {0: 'No', 1: 'Sí'},
    'Dislipemia':           {0: 'No', 1: 'Sí'},
    'Fumador_tabaco':       {0: 'No', 1: 'Sí', 2:'Exfumador'},
    'lado':                 {0: 'Derecho', 1: 'Izquierdo', 2:'Bilateral', 3:'Trasplante'},
    'CATETERJJPREVIO':      {0: 'No', 1: 'Sí'},
    'Dilatacion_ureteral':  {0: 'No', 1: 'Sí'},
    'DERIVACIONPOSTERIOR':  {0: 'No', 1: 'Mono J', 2:'Doble J'},
    'cultivo':              {0: 'No', 1: 'Sí'},
    'Cultresul':            {0: 'Negativo', 1: 'Negativo', 2:'Contaminado'},
    'Microorganismo':       {0: 'Negativo', 1: 'Ureasa negativo', 2:'Ureasa positivo'},
    'ANOMALIAANATÓMICA':    {0: 'No', 1: 'Sí'},
    'TIPOANOMALIA':         {1: 'Divertículo calicial', 2: 'Estenosis ureteral', 3: 'EUPU', 4: 'Riñón en herradura', 5: 'Duplicidad ureteral incompleta', 6: 'Malrotación renal', 7: 'Estenosis de uretra', 8: 'Ectopia renal', 9: 'Duplicidad ureteral completa', 10: 'Trasplante renal'},
    'tipotto':              {0: 'Primario', 1: 'Post-ESWL', 2: 'Post-URS', 3: 'Post-NLP'},
    'GRUPO':                {1: '<2-1,5cm', 2: '>=2-1,5cm'},
    'LOCAL_COD':            {0: 'ureteral', 1: 'renal', 2:'ambas'},
    'LOCALIZACIÓN':         {1: 'Pelvis renal', 2: 'GCS', 3:'GCM', 4:'GCI'},
    'Calibre_vaina':        {0: '9-11', 1: '10-12'},
    'Procalcitonina':       {0: 'Rango normal', 1: 'Fuera del rango'},
    'LeucocitosTotales':    {0: 'Rango normal', 1: 'Fuera del rango'},
    'ProcentajeNeutrofilos':{0: 'Rango normal', 1: 'Fuera del rango'},
    #'ASA':                  {1: 'I', 2: 'II', 3: 'III', 4: 'IV'},
    #'GUY_SCORE':            {1: 'I', 2: 'II', 3: 'III', 4: 'IV'},
    'Vaina_acceso':         {0: 'No', 1: 'Sí'},
    #'PULS':                 {0: 'No', 1: 'Sí'},
}

In [ ]:
import pandas as pd

# ignorando NaNs igual que lo hace la SPSS
categoricas = [
    col for col in patient_data.columns
    if col not in numericas and col not in ['codigo']
]

# Construir tabla resumen en formato compacto con moda
filas = []

for col in categoricas:
    abs_freq = patient_data[col].value_counts(dropna=True)
    rel_freq = patient_data[col].value_counts(dropna=True, normalize=True) * 100
    
    resumen = pd.DataFrame({'n': abs_freq, '%': rel_freq.round(1)})
    
    # Calcular moda (puede haber más de una)
    moda_vals = patient_data[col].mode().tolist()
    
    # Aplicar mapeo si existe
    if col in mapeos:
        resumen.index = resumen.index.map(lambda x: mapeos[col].get(x, x))
        moda_vals = [mapeos[col].get(v, v) for v in moda_vals]
    
    moda_str = " / ".join(str(v) for v in moda_vals)
    
    # Construir string de frecuencias en formato "Valor: X%"
    freq_str = "  ".join(f"{val}: {row['%']}%" for val, row in resumen.iterrows())
    
    filas.append({
        'Variable': col,
        'Frequency per Possible Value': freq_str,
        'Mode': moda_str
    })

# Crear DataFrame final
tabla_final = pd.DataFrame(filas, columns=['Variable', 'Frequency per Possible Value', 'Mode'])

# Mostrar sin índice, con máximo ancho de columna
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
print(tabla_final.to_string(index=False))

In [ ]:
filas = []

for col in categoricas:
    abs_freq = patient_data[col].value_counts(dropna=False)
    rel_freq = patient_data[col].value_counts(dropna=False, normalize=True) * 100

    for valor, n in abs_freq.items():
        etiqueta = mapeos[col].get(valor, valor) if col in mapeos else valor
        filas.append({
            'Variable': col,
            'Categoría': etiqueta,
            'n': n,
            '%': round(rel_freq[valor], 2)
        })

tabla = pd.DataFrame(filas)
tabla

In [ ]:
# ignorando NaNs igual que lo hace la SPSS
categoricas = [
    col for col in patient_data.columns
    if col not in numericas and col not in ['codigo']
]
for col in categoricas:
    print(f"\n--- {col} ---")
    
    # Conteo absoluto y porcentaje ignorando NaNs
    abs_freq = patient_data[col].value_counts(dropna=True)
    rel_freq = patient_data[col].value_counts(dropna=True, normalize=True) * 100
    
    resumen = pd.DataFrame({'n': abs_freq, '%': rel_freq.round(2)})
    
    # Aplicar mapeo si existe
    if col in mapeos:
        resumen.index = resumen.index.map(lambda x: mapeos[col].get(x, x))
    
    print(resumen)

##### update

In [ ]:

mapeos = {
    'RESUELTOEN6m':         {0: 'No', 1: 'Sí'},
    'Genero':               {0: 'Hombre', 1: 'Mujer'},
    'Obesidad':             {0: 'No', 1: 'Sí'},
    'Diabetes':             {0: 'No', 1: 'DM insulina', 2: 'DM sin insulina'},
    'HTA':                  {0: 'No', 1: 'Sí'},
    'Dislipemia':           {0: 'No', 1: 'Sí'},
    'Fumador_tabaco':       {0: 'No', 1: 'Sí', 2:'Exfumador'},
    'lado':                 {0: 'Derecho', 1: 'Izquierdo', 2:'Bilateral', 3:'Trasplante'},
    'CATETERJJPREVIO':      {0: 'No', 1: 'Sí'},
    'Dilatacion_ureteral':  {0: 'No', 1: 'Sí'},
    'DERIVACIONPOSTERIOR':  {0: 'No', 1: 'Mono J', 2:'Doble J'},
    'cultivo':              {0: 'No', 1: 'Sí'},
    'Cultresul':            {0: 'Negativo', 1: 'Positivo', 2:'Contaminado'},
    'Microorganismo':       {0: 'Negativo', 1: 'Ureasa negativo', 2:'Ureasa positivo'},
    'ANOMALIAANATÓMICA':    {0: 'No', 1: 'Sí'},
    'TIPOANOMALIA':         {1: 'Divertículo calicial', 2: 'Estenosis ureteral', 3: 'EUPU', 4: 'Riñón en herradura', 5: 'Duplicidad ureteral incompleta', 6: 'Malrotación renal', 7: 'Estenosis de uretra', 8: 'Ectopia renal', 9: 'Duplicidad ureteral completa', 10: 'Trasplante renal'},
    'tipotto':              {0: 'Primario', 1: 'Post-ESWL', 2: 'Post-URS', 3: 'Post-NLP'},
    'GRUPO':                {1: '<2-1,5cm', 2: '>=2-1,5cm'},
    'LOCAL_COD':            {0: 'ureteral', 1: 'renal', 2:'ambas'},
    'LOCALIZACIÓN':         {1: 'Pelvis renal', 2: 'GCS', 3:'GCM', 4:'GCI'},
    'Calibre_vaina':        {0: 'tamaño 1', 1: 'tamaño 2', 2: 'tamaño 3'}, # {0: '9-11', 1: '10-12'},
    'Procalcitonina':       {0: 'Rango normal', 1: 'Fuera del rango'},
    'LeucocitosTotales':    {0: 'Rango normal', 1: 'Fuera del rango'},
    'ProcentajeNeutrofilos':{0: 'Rango normal', 1: 'Fuera del rango'},
    #'ASA':                  {1: 'I', 2: 'II', 3: 'III', 4: 'IV'},
    #'GUY_SCORE':            {1: 'I', 2: 'II', 3: 'III', 4: 'IV'},
    'Vaina_acceso':         {0: 'No', 1: 'Sí'},
    'PULS':                 {0: 'Sin lesión', 1: 'Edema leve o hiperemia', 2: 'Lesión de la mucosa o abrasiones leves', 3: 'Lesiones profundas de la mucosa, perforación sin extravasación',
                            4: 'Perforación ureteral con extravasación de líquido o avulsión', 5: 'Rotura o desgarro completo del uréter'}
}

In [ ]:
categoricas

In [ ]:
import matplotlib.pyplot as plt

categoricas_ = [
    col for col in categoricas
    if col not in ['tipotto','LOCAL_COD','GUY_SCORE','Dilatacion_ureteral','Vaina_acceso','Calibre_vaina','PULS']
    and col not in ['RESUELTOEN6m','Complicaciones','tipocomplicacion','complicaciones_intra','complicaciones_post','trasfusion','complicaclavien','reintervcomplic','embolizacion','SANGRADO','PERFORACION',
    'INFECCION','SEPSIS','HEMATURIA','HEMATOMA','IngresoSiNo']
]
n_cols = 2
n_rows = (len(categoricas_) + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(14, 3 * n_rows)
)

axes = axes.flatten()

for ax, col in zip(axes, categoricas_):

    data = patient_data[col].dropna()

    if col in mapeos:
        data = data.map(mapeos[col]).dropna()

    # Unir categorías en TIPOANOMALIA
    if col == "TIPOANOMALIA":
        data = data.replace({
            "Estenosis ureteral": "Estenosis",
            "Estenosis de uretra": "Estenosis"
        })

    counts = data.value_counts()

    if counts.empty:
        ax.set_visible(False)
        continue

    total = counts.sum()
    percentages = (counts / total * 100).round(1)

    wedges, _ = ax.pie(
        counts.values,
        labels=None,
        startangle=90
    )

    legend_labels = [
        f"{cat} ({pct}%)"
        for cat, pct in zip(counts.index.astype(str), percentages)
    ]

    ax.legend(
        wedges,
        legend_labels,
        loc="center left",
        bbox_to_anchor=(1, 0.5),
        fontsize=10,
        frameon=False,
        #prop={'weight': 'bold'}
    )

    ax.set_title(col, fontsize=12, fontweight='bold')

# eliminar ejes vacíos
for i in range(len(categoricas_), len(axes)):
    fig.delaxes(axes[i])

plt.subplots_adjust(
    left=0.05,
    right=0.88,
    top=0.95,
    bottom=0.05,
    wspace=-0.3,
    hspace=0.2
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

categoricas_ = ['tipotto','LOCAL_COD','GUY_SCORE','Dilatacion_ureteral','Vaina_acceso','Calibre_vaina','PULS']

n_cols = 2
n_rows = (len(categoricas_) + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(14, 3 * n_rows)
)

axes = axes.flatten()

for ax, col in zip(axes, categoricas_):

    data = patient_data[col].dropna()

    if col in mapeos:
        data = data.map(mapeos[col]).dropna()

    counts = data.value_counts()

    if counts.empty:
        ax.set_visible(False)
        continue

    total = counts.sum()
    percentages = (counts / total * 100).round(1)

    wedges, _ = ax.pie(
        counts.values,
        labels=None,
        startangle=90
    )

    legend_labels = [
        f"{cat} ({pct}%)"
        for cat, pct in zip(counts.index.astype(str), percentages)
    ]

    ax.legend(
        wedges,
        legend_labels,
        loc="center left",
        bbox_to_anchor=(1, 0.5),
        fontsize=10,
        frameon=False,
        #prop={'weight': 'bold'}
    )

    ax.set_title(col, fontsize=12, fontweight='bold')

# eliminar ejes vacíos
for i in range(len(categoricas_), len(axes)):
    fig.delaxes(axes[i])

plt.subplots_adjust(
    left=0.05,
    right=0.88,
    top=0.95,
    bottom=0.05,
    wspace=-0.3,
    hspace=0.2
)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Mapeo 1 -> Sí, 0 -> No
serie = patient_data["RESUELTOEN6m"].map({1: "Sí", 0: "No"})

# Ignorar NaN (listwise deletion estilo SPSS)
serie = serie.dropna()

# Conteo
counts = serie.value_counts()

# Pie chart
plt.figure(figsize=(6,6))
plt.pie(
    counts,
    labels=counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Éxito")

# Leyenda centrada con fondo blanco semitransparente
plt.legend(
    title="Éxito",
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    frameon=True,
    facecolor="white",
    framealpha=0.7
)

plt.show()

## p-valor

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt

feat_cat = 'RESUELTOEN6m'

n_cols = 3
n_rows = -(-len(numericas) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

for idx, feature in enumerate(numericas):
    subset = patient_data[[feat_cat, feature]].dropna()

    num_cat0 = subset.loc[subset[feat_cat] == 0, feature]
    num_cat1 = subset.loc[subset[feat_cat] == 1, feature]

    if num_cat0.empty or num_cat1.empty:
        axes[idx].set_visible(False)
        continue

    axes[idx].boxplot([num_cat0, num_cat1], labels=['No Éxito', 'Éxito'])
    axes[idx].set_title(f"Distribución de {feature}")

    stat, p_value = stats.mannwhitneyu(num_cat0, num_cat1, alternative='two-sided')
    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
    axes[idx].set_xlabel(f"Mann-Whitney p={p_value:.4f} {sig}")

for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(h_pad=2)
plt.show()

In [ ]:
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import math

# =========================================================
# CONFIG
# =========================================================
patient_data = patient_data.loc[:, ~patient_data.columns.duplicated()]

feat_cat1 = 'RESUELTOEN6m'

vars_separadas = ['TIPOANOMALIA']

results = []

# =========================================================
# FUNCIÓN PVALOR
# =========================================================
def format_pvalue(p):
    if p < 0.0001:
        return "<0.0001"
    return f"{p:.4f}"

# =========================================================
# FUNCIÓN VALIDACIÓN
# =========================================================
def variable_valida(df, yvar, xvar):

    if yvar == xvar:
        return False

    if xvar not in df.columns:
        return False

    subset = df[[yvar, xvar]].dropna()

    if subset.empty:
        return False

    if subset[xvar].nunique() < 2:
        return False

    tabla = pd.crosstab(
        subset[yvar],
        subset[xvar]
    )

    if tabla.shape[0] < 2 or tabla.shape[1] < 2:
        return False

    return True

# =========================================================
# FUNCIÓN HEATMAP
# =========================================================
def plot_contingencias(lista_variables):

    valid_vars = [
        v for v in lista_variables
        if variable_valida(patient_data, feat_cat1, v)
    ]

    if len(valid_vars) == 0:
        return

    n_cols = 2
    n_rows = math.ceil(len(valid_vars) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(14, n_rows * 4)
    )

    # Si solo hay 1 fila
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    axes = axes.flatten()

    for i, var in enumerate(valid_vars):

        subset = patient_data[[feat_cat1, var]].dropna().copy()

        # Mapeos
        subset[feat_cat1] = subset[feat_cat1].map(
            mapeos.get(feat_cat1, lambda x: str(x))
        )

        subset[var] = subset[var].map(
            mapeos.get(var, lambda x: str(x))
        )

        # Tablas
        raw_crosstab = pd.crosstab(
            subset[feat_cat1],
            subset[var]
        )

        norm_crosstab = pd.crosstab(
            subset[feat_cat1],
            subset[var],
            normalize='index'
        ) * 100

        # Chi-cuadrado
        _, pvalue, _, _ = chi2_contingency(raw_crosstab)

        # Heatmap
        sns.heatmap(
            norm_crosstab,
            ax=axes[i],
            annot=raw_crosstab,
            fmt='d',
            cmap='Blues',
            linewidths=0.5,
            cbar=False,
            vmin=0,
            vmax=100
        )

        axes[i].set_title(
            f"{var}\np={format_pvalue(pvalue)}",
            fontsize=10
        )

        axes[i].set_xlabel('')
        axes[i].set_ylabel(feat_cat1)

        # Rotación etiquetas
        axes[i].tick_params(axis='x')

        # Guardar resultados
        results.append({
            'Variable': var,
            'p-valor_num': pvalue,
            'p-valor': format_pvalue(pvalue)
        })

    # Ocultar ejes sobrantes
    for j in range(len(valid_vars), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

# =========================================================
# VARIABLES PRINCIPALES
# =========================================================
categoricas_main = [
    c for c in categoricas
    if c not in vars_separadas
    and c not in columnas_complicaciones
]

# =========================================================
# FIGURA PRINCIPAL
# =========================================================
plot_contingencias(
    categoricas_main
)

# =========================================================
# FIGURA SECUNDARIA
# =========================================================
plot_contingencias(
    vars_separadas
)

# =========================================================
# RESULTADOS
# =========================================================
results_df = pd.DataFrame(results)

results_df = (
    results_df
    .sort_values(by='p-valor_num')
    .drop(columns='p-valor_num')
    .reset_index(drop=True)
)

results_df

In [ ]:
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import math

# =========================================================
# CONFIG
# =========================================================
patient_data = patient_data.loc[:, ~patient_data.columns.duplicated()]

feat_cat1 = 'RESUELTOEN6m'

vars_separadas = ['TIPOANOMALIA']

results = []

# =========================================================
# FUNCIÓN PVALOR
# =========================================================
def format_pvalue(p):
    if p < 0.0001:
        return "<0.0001"
    return f"{p:.4f}"

# =========================================================
# FUNCIÓN VALIDACIÓN
# =========================================================
def variable_valida(df, yvar, xvar):

    if yvar == xvar:
        return False

    if xvar not in df.columns:
        return False

    subset = df[[yvar, xvar]].dropna()

    if subset.empty:
        return False

    if subset[xvar].nunique() < 2:
        return False

    tabla = pd.crosstab(
        subset[yvar],
        subset[xvar]
    )

    if tabla.shape[0] < 2 or tabla.shape[1] < 2:
        return False

    return True

# =========================================================
# FUNCIÓN HEATMAP
# =========================================================
def plot_contingencias(lista_variables):

    valid_vars = [
        v for v in lista_variables
        if variable_valida(patient_data, feat_cat1, v)
    ]

    n_cols = 2
    n_rows = math.ceil(len(valid_vars) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(14, n_rows * 4)
    )

    # Si solo hay 1 fila
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    axes = axes.flatten()

    for i, var in enumerate(valid_vars):

        subset = patient_data[[feat_cat1, var]].dropna().copy()

        # Mapeos
        subset[feat_cat1] = subset[feat_cat1].map(
            mapeos.get(feat_cat1, lambda x: str(x))
        )

        subset[var] = subset[var].map(
            mapeos.get(var, lambda x: str(x))
        )

        # Tablas
        raw_crosstab = pd.crosstab(
            subset[feat_cat1],
            subset[var]
        )

        norm_crosstab = pd.crosstab(
            subset[feat_cat1],
            subset[var],
            normalize='index'
        ) * 100

        # Chi2
        _, pvalue, _, _ = chi2_contingency(raw_crosstab)

        # Heatmap
        sns.heatmap(
            norm_crosstab,
            ax=axes[i],
            annot=raw_crosstab,
            fmt='d',
            cmap='Blues',
            linewidths=0.5,
            cbar=False,
            vmin=0,
            vmax=100
        )

        axes[i].set_title(
            f"{var}\np={format_pvalue(pvalue)}",
            fontsize=10
        )

        axes[i].set_xlabel('')
        axes[i].set_ylabel(feat_cat1)

        # Rotación etiquetas
        axes[i].tick_params(axis='x') #, rotation=45

        results.append({
            'Variable': var,
            'p-valor': pvalue
        })

    # Ocultar ejes sobrantes
    for j in range(len(valid_vars), len(axes)):
        axes[j].axis('off')


    plt.tight_layout()
    plt.show()

# =========================================================
# VARIABLES PRINCIPALES
# =========================================================
categoricas_main = [
    c for c in categoricas
    if c not in vars_separadas
    and c not in columnas_complicaciones
]

# =========================================================
# FIGURA PRINCIPAL
# =========================================================
plot_contingencias(
    categoricas_main
)

# =========================================================
# FIGURA SECUNDARIA
# =========================================================
plot_contingencias(
    vars_separadas
)

# =========================================================
# RESULTADOS
# =========================================================
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='p-valor'
)

results_df

## Matriz de correlacion

In [ ]:
import numpy as np

patient_data['ComplInfSeps'] = np.where(
    patient_data[['INFECCION', 'SEPSIS']].isna().any(axis=1),
    np.nan,
    ((patient_data['INFECCION'] == 1) | (patient_data['SEPSIS'] == 1)).astype(int)
)# Contar cuántas veces aparece el valor 1 en las columnas específicas
columnas = ["INFECCION", "SEPSIS", "ComplInfSeps"]
conteos = patient_data[columnas].apply(lambda col: (col == 1).sum())
# Mostrar los resultados
print(conteos)

In [ ]:
import numpy as np

cols = ['HEMATURIA', 'HEMATOMA', 'SANGRADO', 'trasfusion', 'embolizacion']

patient_data['ComplHemo'] = np.where(
    patient_data[cols].isna().any(axis=1),
    np.nan,
    (
        (patient_data['HEMATURIA'] == 1) |
        (patient_data['HEMATOMA'] == 1) |
        (patient_data['SANGRADO'] == 1) |
        (patient_data['trasfusion'] == 1) |
        (patient_data['embolizacion'] == 1)
    ).astype(int)
)
# Contar cuántas veces aparece el valor 1 en las columnas específicas
columnas = ['HEMATURIA','HEMATOMA','SANGRADO','trasfusion','embolizacion','ComplHemo']
conteos = patient_data[columnas].apply(lambda col: (col == 1).sum())
# Mostrar los resultados
print(conteos)

In [ ]:
columnas_complicaciones.remove('IngresoSiNo')
patient_data.drop(columns=columnas_complicaciones, inplace=True)

##### matriz de correlacion OLD

In [ ]:
import seaborn as sns
# Calcular la matriz de correlación
sacar_correlacion = patient_data.drop(columns=['codigo'])
correlacion = sacar_correlacion.corr()
# Visualizar la matriz de correlación con un heatmap
plt.figure(figsize=(20, 10))
sns.heatmap(correlacion, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación')
plt.show()

In [ ]:
# Ver varianza de cada columna (0 = valor constante)
print(patient_data.var(numeric_only=True).sort_values())
# Ver % de NaNs por columna
print((patient_data.isnull().mean() * 100).sort_values(ascending=False).round(2))

## Complicaciones

In [ ]:
patient_data.columns

In [ ]:
patient_data.drop(columns=['IngresoSiNo'], inplace=True)

In [ ]:
patient_data.drop(columns=['ComplInfSeps','ComplHemo'], inplace=True)

In [ ]:
patient_data.columns

Variable anestesia no tiene varianza y todos los valores son "1" ('Vaina_acceso'), por lo cual se elimina del estudio

In [ ]:
# Verificar si la columna tiene solo NaN
patient_data['Vaina_acceso'].isna().sum()

# Verificar si todos los valores son iguales
patient_data['Vaina_acceso'].nunique()
patient_data = patient_data.drop(columns=['Vaina_acceso'])

categoricas.remove('Vaina_acceso')

# Tratamiento de NaNs

**Eliminación de muestras y caracteristicas con una gran ausencia de datos:**

Cantidad de NaNs por columna:

In [ ]:
patient_data.isna().sum()

In [ ]:
data_nan = np.isnan(patient_data.drop(columns=['codigo']))
nans_per_colum = np.sum(data_nan, axis=0)
plt.xticks(rotation=90)
plt.bar(patient_data.drop(columns=['codigo']).columns, nans_per_colum)
plt.show()

Cantidad de NaNs por fila (elemento):

In [ ]:
# Contar los NaNs por fila
nans_por_fila = patient_data.isna().sum(axis=1)
# Filtrar solo las filas que tienen más de 0 NaNs
nans_por_fila_con_na = nans_por_fila[nans_por_fila > 0]
#print(nans_por_fila_con_na)
print('Total de elementos con al menos un NaN:',len(nans_por_fila_con_na)) 
cantidad_por_nans = nans_por_fila.value_counts().sort_index()
print('La cantidad de Nans en una fila y el número de filas con esta cantidad:')
print(cantidad_por_nans)

"Una regla común es eliminar pacientes que tengan más de un 30-40% de datos faltantes."

In [ ]:
len(patient_data.columns)

In [ ]:
#umbral_eliminar_filas_por_Nan = 0.3*len(patient_data.columns) #quizas cuando haya mas car-cas y la distribucion de NaNs sea diferente cambiar el umbral 0,4*
umbral_eliminar_filas_por_Nan =11

Se eliminan los pacientes que tengan un número de NaNs mayor que umbral (30%):

In [ ]:
nans_per_row = np.sum(data_nan, axis=1)
index_rows = np.argwhere(nans_per_row.to_numpy() > umbral_eliminar_filas_por_Nan)[:,0]
patient_data_clean = patient_data.drop(patient_data.index[index_rows])
N_pat, N_columns = patient_data_clean.shape
print(f"Number of patients: {N_pat}")
print(f"Number of columns: {N_columns}")

eliminar los que tienen NaNs en etiqueta

In [ ]:
patient_data_clean.dropna(subset=['RESUELTOEN6m'],inplace=True)
patient_data_clean.isna().sum()

In [ ]:
len(patient_data_clean)

In [ ]:
((patient_data_clean.isna().sum() / len(patient_data_clean)) * 100).round(2)

Eliminar las columnas con NaNs excesivos

In [ ]:
threshold = 0.55  # 55% NaNs para no perder el tipo de anomalia, al cual se le aplicara el one-hot encoding

na_ratio = patient_data_clean.isnull().mean()

cols_to_keep = na_ratio[na_ratio < threshold].index.tolist()

# asegurar que TIPOANOMALIA siempre esté
if "TIPOANOMALIA" not in cols_to_keep:
    cols_to_keep.append("TIPOANOMALIA")

patient_data_clean = patient_data_clean[cols_to_keep]

In [ ]:
# ahora hago el drop de procalcitonina, porque establezco el umbral de 37% para valores faltntes en una variable
# este umbral (37%) principalmente para no perder la varibale de UH que puede ser importante para el exito, al mismo timepo se conservara la variable de lado
patient_data_clean.drop(columns=['Procalcitonina'], inplace=True)


**Vuelvo a quitar decimal sobrante de variables categoricas**

In [ ]:
print(patient_data_clean.columns.tolist())

In [ ]:
# quitar decimal sobrante de variables categoricas
# ###categorias = ['Sexo', 'Diabetes', 'ASA', 'tipotto', 'numero', 'Guy_Score',  'cateterismo_ureteral', 'contraste', 'multitrayecto', 'facilidad_localizacion', 'tubeless', 'Procalcitonina', 'LeucocitosTotales', 'ProcentajeNeutrofilos', 'IngresoSiNo', 'BMI_missing', 'Cultresul_missing', 'Microorganismo_missing', 'diametro_mayor_missing', 'diametro_menor_missing', 'numero_missing', 'localizacion_missing', 'Guy_Score_missing', 'UH_missing', 'distanciapielcalculo_missing', 'tecnica_missing', 'posicion_paciente_missing', 'acceso_puncion_missing', 'contraste_missing', 'metodo_dilatacion_missing', 'multitrayecto_missing', 'calibre_vaina_missing', 'fuente_fragmentacion_missing', 'facilidad_localizacion_missing', 'duracion_tto_missing', 'Cultresul_0.0', 'Cultresul_1.0', 'Cultresul_2.0', 'Microorganismo_0.0', 'Microorganismo_1.0', 'Microorganismo_2.0', 'lado_0', 'lado_1', 'lado_2', 'lado_3', 'localizacion_0.0', 'localizacion_1.0', 'localizacion_2.0', 'localizacion_3.0', 'localizacion_4.0', 'localizacion_5.0', 'localizacion_6.0', 'localizacion_7.0', 'localizacion_8.0', 'localizacion_9.0', 'localizacion_10.0', 'localizacion_11.0', 'localizacion_12.0', 'tecnica_0.0', 'tecnica_1.0', 'tecnica_2.0', 'tecnica_3.0', 'tecnica_4.0', 'tecnica_5.0', 'tecnica_6.0', 'posicion_paciente_0.0', 'posicion_paciente_1.0', 'acceso_puncion_0.0', 'acceso_puncion_1.0', 'acceso_puncion_2.0', 'metodo_dilatacion_0.0', 'metodo_dilatacion_1.0', 'metodo_dilatacion_2.0', 'metodo_dilatacion_3.0', 'fuente_fragmentacion_0.0', 'fuente_fragmentacion_1.0', 'fuente_fragmentacion_2.0', 'fuente_fragmentacion_3.0', 'fuente_fragmentacion_4.0', 'fuente_fragmentacion_5.0', 'fuente_fragmentacion_6.0', 'fuente_fragmentacion_7.0', 'drenajes_0.0', 'drenajes_1.0', 'drenajes_2.0', 'drenajes_3.0', 'drenajes_4.0', 'drenajes_5.0']	
# ### 
# categorias = ['Sexo', 'HTA', 'Fumador_tabaco','Diabetes', 'ASA', 'Cultresul', 'Microorganismo', 'tipotto', 'lado', 'numero', 'localizacion', 'Guy_Score', 'EDAD', 'tecnica', 'posicion_paciente', 'acceso_puncion', 'cateterismo_ureteral', 'contraste', 'metodo_dilatacion', 'multitrayecto', 'fuente_fragmentacion', 'facilidad_localizacion', 'duracion_tto', 'drenajes', 'tubeless', 'Procalcitonina', 'LeucocitosTotales', 'ProcentajeNeutrofilos']
categorias = ['Genero', 'lado', 'cultivo', 'Cultresul', 'Microorganismo','Diabetes', 'HTA', 'Dislipemia', 'Fumador_tabaco', 'ASA', 'ANOMALIAANATÓMICA', 'TIPOANOMALIA',  'LOCALIZACIÓN', 'GRUPO',  'CATETERJJPREVIO',   'DERIVACIONPOSTERIOR',  'LeucocitosTotales', 'ProcentajeNeutrofilos'] # Procalcitonina', # 'tipotto','LOCAL_COD','GUY_SCORE','Dilatacion_ureteral','Calibre_vaina','PULS',
patient_data_clean[categorias] = patient_data_clean[categorias].astype('Int64')
patient_data_clean.head()

# Tratamiento de variables categoricas

Ordenar bien la variable categorica de Diabetes para poder utilizarla como ordinal y no nominal:

Originalmente:
0 NO
1 DM INSULINO
2 DM NO INSULINO

Nuevo:
0 NO
1 DM NO INSULINO
2 DM INSULINO

In [ ]:
patient_data_clean["Diabetes"] = patient_data_clean["Diabetes"].replace({1: 2, 2: 1})

In [ ]:
patient_data_clean["Diabetes"].value_counts()

One-hot-encoding: (mirar cuales son nominales y cuales son ordinales)

In [ ]:
cat_nominales = [ 'lado','cultivo', 'Cultresul', 'Microorganismo','Fumador_tabaco', 'TIPOANOMALIA', 'LOCALIZACIÓN', 'DERIVACIONPOSTERIOR']
patient_data_clean = pd.get_dummies(
    patient_data_clean,
    columns=cat_nominales,
    dtype=int
)

In [ ]:
patient_data_clean.head()

In [ ]:
print(patient_data_clean.columns)

In [ ]:
patient_data_clean['NÚMEROLITIASIS'].unique()

In [ ]:
frecuencias = (
    patient_data_clean['NÚMEROLITIASIS']
    .value_counts(dropna=False)
    .reset_index()
)

frecuencias.columns = ['NÚMEROLITIASIS', 'repeticiones']

print(frecuencias)

In [ ]:
patient_data_clean.loc[patient_data_clean['NÚMEROLITIASIS'] == 0, patient_data_clean.columns[:15]]

**Arreglar acceso_puncion**

In [ ]:
# Si LOCAL_COD_2 == 1 (ambas tecnicas), asignar 1 a las otras dos columnas
# patient_data_clean.loc[patient_data_clean['LOCAL_COD_2'] == 1, ['LOCAL_COD_0', 'LOCAL_COD_1']] = 1
# patient_data_clean.drop(columns='LOCAL_COD_2', inplace=True)

In [ ]:
#patient_data_clean.to_excel("23_patient_data_clean.xlsx", index=False)

# Data splitting

Tratamiento de pacientes repetidos
(from Lab1 script:"In cases with multiple measures for each patient (each patient is present in multiple 
rows), each patient must belong to a single set. Measures of the same patient cannot 
be present on multiple sets (training, validation or test). After the data splitting, remove 
the patient-ID column.")

In [ ]:
# Contar cuántas veces aparece cada ID de paciente en la columna 'n_historia'
repeated_patients = patient_data_clean['codigo'].value_counts()
# Filtrar solo aquellos que se repiten más de una vez (mayor a 1)
repeated_patients = repeated_patients[repeated_patients > 1]
# Contar la frecuencia de cada número de repeticiones
repetition_counts = repeated_patients.value_counts().sort_index()
# Mostrar cuántos pacientes hay con 2 repeticiones, 3 repeticiones, etc.
print(repetition_counts)
# Mostrar la suma total de pacientes repetidos
print("Total de pacientes repetidos:", repetition_counts.sum())

Separación de datos en train, validation y test 70%/15%/15%. Los mismos pacientes siempre están en el mismo set.
También se utiliza StratifiedGroupKFold para garantizar que la proporción entre clases IngresoSi y IngresoNo sea igual en los 3 subconjuntos, es decir, la **ESTRATIFICACIÓN**.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
# split 80% y 20%
# Variables 
X = patient_data_clean.drop(columns=['RESUELTOEN6m', 'codigo'])
y = patient_data_clean['RESUELTOEN6m']
groups = patient_data_clean['codigo'].values

# 5 folds -> 1 fold test = 20%
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=5 #5 #9
)

# Primer split
train_idx, test_idx = next(sgkf.split(X, y, groups))

# Train/Test
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups[train_idx]
groups_test = groups[test_idx]

print("Train size:", len(X_train))
print("Test size:", len(X_test))

(X_train, X_test, y_train, y_test, groups_train, groups_test)

In [ ]:
#Visualización del resultado y comprobación de ausencia de n_historia en los sets:
X_test.head()

In [ ]:
# Crear DataFrame de entrenamiento, validación y test
train_data = pd.concat([X_train, y_train], axis=1)
# val_data = pd.concat([X_val, y_val], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)
train_data.head()

# EDA

**Variables categoricas sin one-hot-encoding**

In [ ]:
# Función para calcular estadísticas descriptivas

def calcular_estadisticas(df, cat_cols):
    stats = {}
    # Para variables categóricas: proporciones
    for col in cat_cols:
        proportions = round(df[col].value_counts(normalize=True) * 100,2)
        for value, proportion in proportions.items():
            stats[f"{col}_{value}"] = {'proportion': proportion}

    return pd.DataFrame(stats).T

# Definir columnas categóricas
one_hot_cols = ['Cultresul','Microorganismo', 'lado','localizacion','tecnica','posicion_paciente','acceso_puncion','metodo_dilatacion','fuente_fragmentacion','drenajes']
#cat_cols = ['Sexo', 'Diabetes', 'ASA', 'tipotto','Guy_Score','IngresoSiNo']
# Unir ambas listas de columnas conocidas
columnas_conocidas = set(one_hot_cols + numericas)
columnas_conocidas.add('codigo')
# Filtrar columnas del DataFrame que no están en las listas
cat_cols = [col for col in patient_data_clean.columns if col not in columnas_conocidas]

# Calcular estadísticas para cada conjunto
stats_train = calcular_estadisticas(train_data, cat_cols)
# stats_valid = calcular_estadisticas(val_data, cat_cols)
stats_test = calcular_estadisticas(test_data, cat_cols)

# Combinar las estadísticas en una tabla
stats_summary = pd.concat([stats_train, stats_test], axis=1, keys=['Train', 'Test'])
# stats_summary = pd.concat([stats_train, stats_valid, stats_test], axis=1, keys=['Train', 'Validation', 'Test'])

print(stats_summary)

In [ ]:
print(stats_summary[-67:])

**Variables numericas**

In [ ]:
# numericas.remove('VOLUMEN')

In [ ]:
# Función para calcular media, desviación estándar y mediana (IQR) con redondeo
def calculate_statistics(df):
    stats = {}
    for column in numericas:
        mean = round(df[column].mean(), 2)
        std = round(df[column].std(), 2)
        median = round(df[column].median(), 2)
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = round(q3 - q1, 2)

        stats[column] = {
            'mean': mean,
            'std': std,
            'median': median,
            'IQR': iqr
        }
    return pd.DataFrame(stats)

# Calcular estadísticas para los tres conjuntos de datos
train_stats = calculate_statistics(train_data)
# validation_stats = calculate_statistics(val_data)
test_stats = calculate_statistics(test_data)
pd.concat([train_stats,test_stats], axis=1, keys=['Train', 'Test'])

**Variables con one-hot encoding**

In [ ]:
# Función para calcular las proporciones de variables después de One-Hot Encoding
def calcular_proporciones_one_hot(df, one_hot_cols):
    proportions = {}
    for col in one_hot_cols:
        # Obtenemos las columnas correspondientes a esta variable después de One-Hot Encoding
        one_hot_columns = [c for c in df.columns if c.startswith(col)]
        
        # Calculamos las proporciones para cada una de las columnas de One-Hot Encoding
        for o_col in one_hot_columns:
            count_ones = df[o_col].sum()  # Contamos cuántos 1s hay en esa columna
            total = df[o_col].shape[0]  # Total de registros en esa columna
            proportion = round((count_ones / total) * 100 ,2 ) # Proporción en porcentaje
            proportions[f"{o_col}_proportion"] = proportion
    
    # Convertimos el diccionario en un DataFrame
    proportions_df = pd.DataFrame(proportions, index=[0])
    return proportions_df

# Ejemplo de uso: Definir columnas One-Hot Encoding
one_hot_cols = cat_nominales
# Calcular estadísticas para cada conjunto
stats_train = calcular_proporciones_one_hot(train_data, one_hot_cols)
# stats_valid = calcular_proporciones_one_hot(val_data, one_hot_cols)
stats_test = calcular_proporciones_one_hot(test_data, one_hot_cols)
df_1hot_eda=pd.concat([stats_train, stats_test], axis=0, keys=['Train', 'Test'])
# df_1hot_eda=pd.concat([stats_train, stats_valid, stats_test], axis=0, keys=['Train', 'Validation', 'Test'])

pd.DataFrame(df_1hot_eda).T

In [ ]:
#!pip install imbalanced-learn

# Imputar sin introducir data leakage

In [ ]:
X_test.isna().sum()

In [ ]:
# Columnas categóricas
# categorical_cols = [col for col in X_train.columns if col not in numericas and col != "RESUELTOEN6m"]
categorical_cols = ['Genero','Diabetes','HTA','Dislipemia','ASA','ANOMALIAANATÓMICA','GRUPO','CATETERJJPREVIO','LeucocitosTotales','ProcentajeNeutrofilos']
# ===== NUMÉRICAS: mediana calculada SOLO en X_train =====
median_values = X_train[numericas].median()

X_train[numericas] = X_train[numericas].fillna(np.round(median_values))
X_test[numericas]  = X_test[numericas].fillna(np.round(median_values))

# ===== CATEGÓRICAS: moda calculada SOLO en X_train =====
mode_values = X_train[categorical_cols].mode().iloc[0]

X_train[categorical_cols] = X_train[categorical_cols].fillna(mode_values)
X_test[categorical_cols]  = X_test[categorical_cols].fillna(mode_values)

In [ ]:
median_values

In [ ]:
mode_values

In [ ]:
X_test.isna().sum()[:-30]

# Entrenar Modelos ML

#### fucnion

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    make_scorer,
    cohen_kappa_score,
    matthews_corrcoef
)

# =========================================================
# FUNCIÓN GENERAL
# =========================================================

def random_search_cv_model(
    model,
    param_dist,
    X_train,
    y_train,
    groups_train,
    refit_metric="auc",
    n_iter=30,
    scale_data=False,
    random_state=42,
    n_jobs=-1,
    verbose=1
):
    """
    Ejecuta RandomizedSearchCV con múltiples métricas.

    Parámetros
    ----------
    model : estimator
        Modelo sklearn (DecisionTree, RandomForest, LogisticRegression, XGBoost, etc.)

    param_dist : dict
        Espacio de hiperparámetros.

    X_train : array-like
    y_train : array-like
    groups_train : array-like

    refit_metric : str
        Métrica usada para seleccionar el mejor modelo.
        Opciones:
        - "auc"
        - "accuracy"
        - "kappa"
        - "mcc"

    scale_data : bool
        Si True, aplica MinMaxScaler mediante Pipeline.
        Útil para Logistic Regression, SVM, etc.

    Returns
    -------
    best_model : estimator
        Mejor modelo ajustado.

    search : RandomizedSearchCV
        Objeto completo del search.
    """

    # =====================================================
    # SCORERS
    # =====================================================

    scoring = {
        "auc": "roc_auc",
        "accuracy": "accuracy",
        "kappa": make_scorer(cohen_kappa_score),
        "mcc": make_scorer(matthews_corrcoef)
    }

    # =====================================================
    # PIPELINE OPCIONAL
    # =====================================================

    if scale_data:

        estimator = Pipeline([
            ("scaler", MinMaxScaler()),
            ("model", model)
        ])

    else:

        estimator = model

    # =====================================================
    # CROSS VALIDATION
    # =====================================================

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state
    )

    # =====================================================
    # RANDOM SEARCH
    # =====================================================

    search = RandomizedSearchCV(
        estimator=estimator,

        param_distributions=param_dist,

        n_iter=n_iter,

        scoring=scoring,

        refit=refit_metric,

        cv=cv,

        n_jobs=n_jobs,

        verbose=verbose,

        random_state=random_state
    )

    # =====================================================
    # ENTRENAMIENTO
    # =====================================================

    search.fit(
        X_train,
        y_train,
        groups=groups_train
    )

    # =====================================================
    # RESULTADOS
    # =====================================================

    best_idx = search.best_index_

    # ===== AUC =====

    mean_auc = search.cv_results_["mean_test_auc"][best_idx]
    std_auc = search.cv_results_["std_test_auc"][best_idx]

    # ===== ACCURACY =====

    mean_acc = search.cv_results_["mean_test_accuracy"][best_idx]
    std_acc = search.cv_results_["std_test_accuracy"][best_idx]

    # ===== KAPPA =====

    mean_kappa = search.cv_results_["mean_test_kappa"][best_idx]
    std_kappa = search.cv_results_["std_test_kappa"][best_idx]

    # ===== MCC =====

    mean_mcc = search.cv_results_["mean_test_mcc"][best_idx]
    std_mcc = search.cv_results_["std_test_mcc"][best_idx]

    print("\n=========================")
    print("RESULTADOS CV")
    print("=========================\n")

    print(f"AUC CV = {mean_auc:.3f} ± {std_auc:.3f}")
    print(f"Accuracy CV = {mean_acc:.3f} ± {std_acc:.3f}")
    print(f"Kappa CV = {mean_kappa:.3f} ± {std_kappa:.3f}")
    print(f"MCC CV = {mean_mcc:.3f} ± {std_mcc:.3f}")

    print("\n=========================")
    print("MEJORES HIPERPARÁMETROS")
    print("=========================\n")

    print(search.best_params_)

    # =====================================================
    # MEJOR MODELO
    # =====================================================

    best_model = search.best_estimator_

    summary = {
        "model": type(model).__name__,

        "refit_metric": refit_metric,

        "auc_mean": mean_auc,
        "auc_std": std_auc,

        "accuracy_mean": mean_acc,
        "accuracy_std": std_acc,

        "kappa_mean": mean_kappa,
        "kappa_std": std_kappa,

        "mcc_mean": mean_mcc,
        "mcc_std": std_mcc,

        "best_params": search.best_params_
    }

    return best_model, search, summary

#### resultados

In [ ]:
results = []

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier()

param_dist = {
    "class_weight": [None, "balanced", {0: 3, 1: 0}, {0:5, 1:0}, {0:10, 1:0}],
    "max_depth": [2, 3, 5, 10, None],
    "min_samples_leaf": [1, 5, 10, 20, 50],
    "min_samples_split": [2, 5, 10, 20]
}

best_dt_model_auc, search_dt_auc, summary_dt_auc = random_search_cv_model(
    model=tree_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="auc",
    scale_data=False
)

results.append(summary_dt_auc)

In [ ]:
best_dt_model_kappa, search_dt_kappa, summary_dt_kappa = random_search_cv_model(
    model=tree_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="kappa",
    scale_data=False
)

results.append(summary_dt_kappa)

In [ ]:
best_dt_model_mcc, search_dt_mcc, summary_dt_mcc = random_search_cv_model(
    model=tree_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="mcc",
    scale_data=False
)

results.append(summary_dt_mcc)

Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)

# =========================
# ESPACIO DE HIPERPARÁMETROS
# =========================

param_dist = {
    "class_weight": [None, "balanced", {0: 3, 1: 0}, {0:5, 1:0}, {0:10, 1:0}],
    "n_estimators": [10, 50, 100, 200, 300],
    "max_depth": [3, 5, 10, 20, None],
    "min_samples_leaf": [1, 5, 10, 15, 20, 50],
    "min_samples_split": [2, 5, 10, 20],
    "max_features": ["sqrt", "log2", None]
}

best_rf_model_auc, search_rf_auc, summary_rf_auc = random_search_cv_model(
    model=rf_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="auc",
    scale_data=False
)

results.append(summary_rf_auc)

In [ ]:
best_rf_model_kappa, search_rf_kappa, summary_rf_kappa = random_search_cv_model(
    model=rf_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="kappa",
    scale_data=False
)

results.append(summary_rf_kappa)

In [ ]:
best_rf_model_mcc, search_rf_mcc, summary_rf_mcc = random_search_cv_model(
    model=rf_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="mcc",
    scale_data=False
)

results.append(summary_rf_mcc)

XGboost

In [ ]:
ratio_real = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
ratio_real

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

param_dist = {
    "scale_pos_weight": [1, 0.33 ], #, 0.2, 0.1, ratio_real],
    
    "n_estimators": [50, 100, 200, 300],
    
    "max_depth": [2, 3, 5, 7, 10],
    
    "learning_rate": [0.001, 0.01, 0.05, 0.1, 0.2],
    
    "min_child_weight": [1, 5, 10, 20, 35],
    
    "subsample": [0.5, 0.7, 0.8, 1.0],
    
    "colsample_bytree": [0.5, 0.7, 0.8, 1.0],
    
    "gamma": [0, 0.1, 0.3, 1],
    
    "reg_alpha": [0, 0.01, 0.1, 1],
    
    "reg_lambda": [0.1, 1, 5, 10]
}

best_xgb_model_auc, search_xgb_auc, summary_xgb_auc = random_search_cv_model(
    model=xgb_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="auc",
    scale_data=False
)

results.append(summary_xgb_auc)

In [ ]:
best_xgb_model_kappa, search_xgb_kappa, summary_xgb_kappa = random_search_cv_model(
    model=xgb_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="kappa",
    scale_data=False
)

results.append(summary_xgb_kappa)

In [ ]:
best_xgb_model_mcc, search_xgb_mcc, summary_xgb_mcc = random_search_cv_model(
    model=xgb_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="mcc",
    scale_data=False
)

results.append(summary_xgb_mcc)

Regresion logistica

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# =====================================================
# MODELO
# =====================================================

logreg_model = LogisticRegression(
    random_state=42,
    max_iter=5000
)

# =====================================================
# HIPERPARÁMETROS
# =====================================================

param_dist = {
    "model__class_weight": [None, "balanced", {0: 3, 1: 0}, {0:5, 1:0}, {0:10, 1:0}],
    "model__C": np.logspace(-4, 4, 20),

    "model__penalty": ["l1", "l2"],

    "model__solver": ["liblinear"]
}

# =====================================================
# RANDOM SEARCH
# =====================================================

best_logreg_model_auc, search_logreg_auc, summary_logreg_auc = random_search_cv_model(
    model=logreg_model,

    param_dist=param_dist,

    X_train=X_train,

    y_train=y_train,

    groups_train=groups_train,

    refit_metric="auc",

    n_iter=30,

    scale_data=True,

    random_state=42,

    n_jobs=-1,

    verbose=1
)

results.append(summary_logreg_auc)

In [ ]:
best_logreg_model_kappa, search_logreg_kappa, summary_logreg_kappa = random_search_cv_model(
    model=logreg_model, param_dist=param_dist, X_train=X_train, y_train=y_train,
    groups_train=groups_train, refit_metric="kappa", n_iter=30, scale_data=True,
    random_state=42, n_jobs=-1, verbose=1)

results.append(summary_logreg_kappa)

In [ ]:
best_logreg_model_mcc, search_logreg_mcc, summary_logreg_mcc = random_search_cv_model(
    model=logreg_model, param_dist=param_dist, X_train=X_train, y_train=y_train,
    groups_train=groups_train, refit_metric="mcc", n_iter=30, scale_data=True,
    random_state=42, n_jobs=-1, verbose=1)

results.append(summary_logreg_mcc)

# Comparar el rendimiento de los modelos entrenados

In [ ]:
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
import pandas as pd

# =====================================================
# DATAFRAME
# =====================================================

results_df = pd.DataFrame(results)

# =====================================================
# RENOMBRAR MODELOS
# =====================================================

model_mapping = {
    "DecisionTreeClassifier": "DT",
    "RandomForestClassifier": "RF",
    "XGBClassifier": "XGB",
    "LogisticRegression": "RegLog"
}

results_df["model"] = results_df["model"].map(model_mapping)

# =====================================================
# REGLOG PRIMERO
# =====================================================

model_order = ["RegLog", "DT", "RF", "XGB" ]

results_df["model"] = pd.Categorical(
    results_df["model"],
    categories=model_order,
    ordered=True
)

results_df = results_df.sort_values(
    by=["model", "refit_metric"]
)

# =====================================================
# REDONDEAR
# =====================================================

metric_cols = [
    "auc_mean",
    "auc_std",
    "accuracy_mean",
    "accuracy_std",
    "kappa_mean",
    "kappa_std",
    "mcc_mean",
    "mcc_std"
]

results_df[metric_cols] = results_df[metric_cols].round(3)

# =====================================================
# CONFIGURACIÓN DISPLAY
# =====================================================

pd.set_option("display.max_columns", None)

pd.set_option("display.width", 200)

pd.set_option("display.max_colwidth", 100)

pd.set_option("display.expand_frame_repr", False)

# =====================================================
# PRINT
# =====================================================

print(results_df)

# Prueba final sobre el test set con el mejor modelo, poninedo el umbral personalizado

#### funcion 2 para guardar resultados en el dataframe

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score
)

# =====================================================
# FUNCIÓN DE EVALUACIÓN GENERAL
# =====================================================

def evaluate_model(
    model,
    X_test,
    y_test,
    model_name="Modelo",
    threshold=0.5,
    plot=True
):

    # ==========================================
    # PROBABILIDADES Y PREDICCIONES
    # ==========================================

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    # ==========================================
    # MATRIZ DE CONFUSIÓN
    # ==========================================

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # ==========================================
    # MÉTRICAS
    # ==========================================

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    sensibilidad = recall_score(y_test, y_pred)
    especificidad = tn / (tn + fp) if (tn + fp) > 0 else 0

    vpp = precision_score(y_test, y_pred, zero_division=0)
    vpn = tn / (tn + fn) if (tn + fn) > 0 else 0

    f1 = f1_score(y_test, y_pred, zero_division=0)

    # ==========================================
    # DICCIONARIO RESULTADOS
    # ==========================================

    metrics_dict = {
        "modelo": model_name,
        "umbral": threshold,
        "auc": round(auc, 4),
        "accuracy": round(accuracy, 4),
        "sensibilidad": round(sensibilidad, 4),
        "especificidad": round(especificidad, 4),
        "vpp": round(vpp, 4),
        "vpn": round(vpn, 4),
        "f1_score": round(f1, 4),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    # ==========================================
    # PRINT RESULTADOS
    # ==========================================

    print("\n" + "=" * 50)
    print(f"EVALUACIÓN DEL MODELO: {model_name}")
    print("=" * 50)

    for key, value in metrics_dict.items():
        if key != "modelo":
            print(f"{key}: {value}")

    # ==========================================
    # GRÁFICOS
    # ==========================================

    if plot:

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # ------------------------------------------------
        # MATRIZ DE CONFUSIÓN
        # ------------------------------------------------

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["Negativo", "Positivo"]
        )

        disp.plot(
            cmap="Blues",
            values_format="d",
            ax=axes[0],
            colorbar=False
        )

        # CAMBIAR ETIQUETAS AL CASTELLANO
        axes[0].set_title(f"Matriz de confusión\n{model_name}")

        axes[0].set_xlabel("Etiqueta predicha")
        axes[0].set_ylabel("Etiqueta real")

        axes[0].grid(False)

        # ------------------------------------------------
        # CURVA ROC
        # ------------------------------------------------

        fpr, tpr, _ = roc_curve(y_test, y_proba)

        axes[1].plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"AUC = {auc:.3f}"
        )

        axes[1].plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            linewidth=1
        )

        axes[1].set_title(f"Curva ROC\n{model_name}")

        axes[1].set_xlabel("Tasa de falsos positivos")
        axes[1].set_ylabel("Tasa de verdaderos positivos")

        axes[1].legend(loc="lower right")
        axes[1].grid(True)

        plt.tight_layout()
        plt.show()

    return metrics_dict

#### Resultados

In [ ]:
results_test = []

In [ ]:
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn import set_config

set_config(enable_metadata_routing=True)

threshold_reglog_model= TunedThresholdClassifierCV(
    estimator=best_logreg_model_kappa,

    scoring="balanced_accuracy",   

    cv=StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
)

threshold_reglog_model.fit(
    X_train,
    y_train,
    groups=groups_train
)

threshold_reglog_model.best_threshold_

In [ ]:
results = evaluate_model(
    model=best_logreg_model_kappa,
    X_test=X_test,
    y_test=y_test, model_name="Regresión Logística",
    threshold=threshold_reglog_model.best_threshold_
)
results_test.append(results)

In [ ]:
results = evaluate_model(
    model=best_logreg_model_kappa,
    X_test=X_test,
    y_test=y_test, model_name="Regresión Logística",
    threshold=0.5
)
results_test.append(results)

In [ ]:
set_config(enable_metadata_routing=True)

threshold_dt_model= TunedThresholdClassifierCV(
    estimator=best_dt_model_kappa,

    scoring="balanced_accuracy",   

    cv=StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
)

threshold_dt_model.fit(
    X_train,
    y_train,
    groups=groups_train
)

threshold_dt_model.best_threshold_

In [ ]:
results = evaluate_model(
    model=best_dt_model_kappa,
    X_test=X_test,
    y_test=y_test, model_name="Árbol de decisión",
    threshold=threshold_dt_model.best_threshold_
)
results_test.append(results)

In [ ]:
results = evaluate_model(
    model=best_dt_model_kappa,
    X_test=X_test,
    y_test=y_test, model_name="Árbol de decisión",
    threshold=0.5
)
results_test.append(results)

In [ ]:
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn import set_config

set_config(enable_metadata_routing=True)

threshold_rf_model_auc = TunedThresholdClassifierCV(
    estimator=best_rf_model_auc,

    scoring="balanced_accuracy",   # maximizar sensibilidad

    cv=StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
)

threshold_rf_model_auc.fit(
    X_train,
    y_train,
    groups=groups_train
)

threshold_rf_model_auc.best_threshold_

In [ ]:
results = evaluate_model(
    model=best_rf_model_auc,
    X_test=X_test,
    y_test=y_test, model_name="Random Forest",
    threshold=threshold_rf_model_auc.best_threshold_
)
results_test.append(results)

In [ ]:
results = evaluate_model(
    model=best_rf_model_auc,
    X_test=X_test,
    y_test=y_test, model_name="Random Forest",
    threshold=0.5
)
results_test.append(results)

In [ ]:
set_config(enable_metadata_routing=True)

threshold_xgb_model= TunedThresholdClassifierCV(
    estimator=best_xgb_model_mcc,

    scoring="balanced_accuracy",   # maximizar sensibilidad

    cv=StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
)

threshold_xgb_model.fit(
    X_train,
    y_train,
    groups=groups_train
)

threshold_xgb_model.best_threshold_

In [ ]:
results = evaluate_model(
    model=best_xgb_model_mcc,
    X_test=X_test,
    y_test=y_test, model_name="XGBoost",
    threshold=threshold_xgb_model.best_threshold_
)
results_test.append(results)

In [ ]:
results = evaluate_model(
    model=best_xgb_model_mcc,
    X_test=X_test,
    y_test=y_test, model_name="XGBoost",
    threshold=0.5
)
results_test.append(results)

In [ ]:
# ==========================================
# DATAFRAME RESUMEN
# ==========================================
summary_df = pd.DataFrame(results_test)
# Ordenar por AUC
summary_df = summary_df.sort_values(
    by="auc",
    ascending=False
)
print(summary_df)

In [ ]:
df_formateado_test = summary_df.iloc[:, :-5].applymap(
    lambda x: f"{x:.4f}".replace(".", ",") if isinstance(x, (int, float)) else x
)
df_formateado_test

La tabla de train:

In [ ]:
df = results_df.iloc[:, :-1].copy()

metricas = ["auc", "accuracy", "kappa", "mcc"]

# empezamos desde el dataframe original pero eliminando columnas mean/std
df_final = df.drop(columns=[f"{m}_mean" for m in metricas] +
                             [f"{m}_std" for m in metricas])

for m in metricas:
    df_final[m] = (
        df[f"{m}_mean"].map(lambda x: f"{x:.3f}".replace(".", ",")) +
        " ± " +
        df[f"{m}_std"].map(lambda x: f"{x:.3f}".replace(".", ","))
    )
df_final

# Shapley

In [ ]:
import re
import shap
# Variables que fueron One-Hot Encoded
ohe_vars = cat_nominales # = [ 'lado','cultivo', 'Cultresul', 'Microorganismo','Fumador_tabaco', 'TIPOANOMALIA', 'LOCALIZACIÓN', 'DERIVACIONPOSTERIOR']


# Crear diccionario de nombres legibles
rename_dict = {}

for col in X_test.columns:

    renamed = False

    for var in ohe_vars:

        # Busca patrones tipo localizacion_0, localizacion_1.0, etc.
        pattern = f"^{re.escape(var)}_(.+)$"
        match = re.match(pattern, col)

        if match:
            valor = match.group(1)

            try:
                valor_num = float(valor)

                # Si existe en el mapeo
                if valor_num in mapeos[var]:
                    rename_dict[col] = f"{var}: {mapeos[var][valor_num]}"
                elif int(valor_num) in mapeos[var]:
                    rename_dict[col] = f"{var}: {mapeos[var][int(valor_num)]}"
                else:
                    rename_dict[col] = col

            except:
                rename_dict[col] = col

            renamed = True
            break

    if not renamed:
        rename_dict[col] = col

# Renombrar columnas
X_test_shap = X_test.rename(columns=rename_dict)

explainer = shap.TreeExplainer(best_rf_model_auc)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(
    shap_values[:, :, 1],
    X_test_shap
)

In [ ]:
# =====================================================
# EXPLICADOR SHAP (Random Forest)
# =====================================================

explainer = shap.TreeExplainer(best_xgb_model_auc)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(
    shap_values,
    X_test_shap
)

-----

-------------------------------------------------------------------------------------------------------------------